In [1]:
from benchmarks import UkrWikiRetrieval, UkrPravdaRetrieval
from mteb import MTEB
from sentence_transformers import SentenceTransformer
import json

import torch
import gc
from sentence_transformers import SentenceTransformer
import pandas as pd
from pathlib import Path


# Run evaluation using MTEB

In [2]:
model_names = [
    "intfloat/multilingual-e5-base",
    # "youscan/ukr-roberta-base",
    "m-rudko-pn/e5-small-ukr-wikipedia",
    "m-rudko-pn/e5-base-ukr-wikipedia",
]


benchmark = MTEB(tasks=[UkrWikiRetrieval(), UkrPravdaRetrieval()])

for model_name in model_names:
    print(f"Running benchmark for {model_name}")
    model = SentenceTransformer(model_name)
    res = benchmark.run(model, verbosity=2, output_folder=f"results/wiki",
                        overwrite_results=False, encode_kwargs={"batch_size": 64})
    print(res)

res

───────────────────────────────────────────────── Selected tasks  ─────────────────────────────────────────────────

Retrieval

- UkrWikiRetrieval, s2p

- UkrPravdaRetrieval, s2p

Overwrite dataset info from restored data version if exists.
INFO:datasets.builder:Overwrite dataset info from restored data version if exists.
Loading Dataset info from /home/imax/.cache/huggingface/datasets/shamotskyi___ukr_pravda_2y/default/0.0.0/be2be302c4c659d362b6fae64d7526cd1901bca6
INFO:datasets.info:Loading Dataset info from /home/imax/.cache/huggingface/datasets/shamotskyi___ukr_pravda_2y/default/0.0.0/be2be302c4c659d362b6fae64d7526cd1901bca6
Found cached dataset ukr_pravda_2y (/home/imax/.cache/huggingface/datasets/shamotskyi___ukr_pravda_2y/default/0.0.0/be2be302c4c659d362b6fae64d7526cd1901bca6)
INFO:datasets.builder:Found cached dataset ukr_pravda_2y (/home/imax/.cache/huggingface/datasets/shamotskyi___ukr_pravda_2y/default/0.0.0/be2be302c4c659d362b6fae64d7526cd1901bca6)
Loading Dataset info from /home/imax/.cache/huggingface/datasets/shamotskyi___ukr_pravda_2y/default/0.0.0/be2be302c4c659d362b6fae64d7526cd1901bca6
INFO:datasets.info:Loading Dataset info from /home/imax/.c

Loaded 61629 documents.


Batches:   0%|          | 0/963 [00:00<?, ?it/s]


KeyboardInterrupt



# View evaluation results

In [3]:

result_root = Path('/home/imax/AllHomework/Golden-Retriever/results/wiki')


def load_results() -> pd.DataFrame:
    results = {}
    for model in result_root.glob('*'):
        for model_result in model.glob('**/*.json'):
            if model_result.stem == 'model_meta':
                continue

            if not model_result.is_file():
                continue

            model_name = model.stem
            if model_name not in results:
                results[model_name] = {}
            with open(model_result, 'r') as json_file:
                results[model_name][model_result.stem] = json.load(json_file)

    result_index = pd.MultiIndex.from_product([['UkrWikiRetrieval', 'UkrPravdaRetrieval'],
                                               ['recall_at_1', 'recall_at_3', 'recall_at_5', 'ndcg_at_1', 'ndcg_at_3',
                                                'ndcg_at_5', ]], names=['task', 'score'])

    df = pd.DataFrame(results).T.reset_index(names=['model_name'])

    result_df = pd.DataFrame(columns=result_index, index=df['model_name']).reset_index(names=['model_name'])

    for dataset in result_index.levels[0]:
        dataset_result_df = pd.json_normalize((pd.json_normalize(df[dataset])['scores.test']).explode())

        for score in result_index.levels[1]:
            result_df[(dataset, score)] = dataset_result_df[score].values
    return result_df.set_index('model_name')

load_results()

task                               UkrWikiRetrieval                          \
score                                   recall_at_1 recall_at_3 recall_at_5   
model_name                                                                    
no_model_name_available                     0.34950     0.42885     0.47230   
m-rudko-pn__e5-base-ukr-wikipedia           0.38886     0.48781     0.53003   
m-rudko-pn__e5-small-ukr-wikipedia          0.34950     0.42885     0.47230   
youscan__ukr-roberta-base                   0.02342     0.04146     0.05266   
intfloat__multilingual-e5-small             0.24394     0.29851     0.32812   
intfloat__multilingual-e5-base              0.25816     0.31544     0.33937   

task                                                              \
score                              ndcg_at_1 ndcg_at_3 ndcg_at_5   
model_name                                                         
no_model_name_available              0.47865   0.49317   0.51018   
m-rudko-pn__e5-base-ukr-wikipedia    0.55346   0.57362   0.57936   
m-rudko-pn__e5-small-ukr-wikipedia   0.47865   0.49317   0.51018   
youscan__ukr-roberta-base            0.03983   0.04760   0.04859   
intfloat__multilingual-e5-small      0.26337   0.28951   0.30110   
intfloat__multilingual-e5-base       0.27915   0.30857   0.31753   

task                               UkrPravdaRetrieval                          \
score                                     recall_at_1 recall_at_3 recall_at_5   
model_name                                                                      
no_model_name_available                           NaN         NaN         NaN   
m-rudko-pn__e5-base-ukr-wikipedia             0.75105     0.83836     0.85557   
m-rudko-pn__e5-small-ukr-wikipedia            0.80493     0.87634     0.89159   
youscan__ukr-roberta-base                     0.13859     0.19474     0.22655   
intfloat__multilingual-e5-small               0.79617     0.87374     0.89387   
intfloat__multilingual-e5-base                0.78351     0.85881     0.87634   

task                                                              
score                              ndcg_at_1 ndcg_at_3 ndcg_at_5  
model_name                                                        
no_model_name_available                  NaN       NaN       NaN  
m-rudko-pn__e5-base-ukr-wikipedia    0.75105   0.80274   0.80987  
m-rudko-pn__e5-small-ukr-wikipedia   0.80493   0.84773   0.85400  
youscan__ukr-roberta-base            0.13859   0.17168   0.18484  
intfloat__multilingual-e5-small      0.79617   0.84261   0.85096  
intfloat__multilingual-e5-base       0.78351   0.82856   0.83576